# 2.1 Utility-Level EDA

Exploratory data analysis for utility-level outage statistics.

Includes:
- Frequency statistics for yearly outages, customer_hours, customer_hours_per_capita
- Seasonality analysis (by month)
- Time of day analysis (by hour)
- Market type breakdown

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load county-level data (which includes utility enrichments)
data_dir = Path("../../data/processed/introduction")

with open(data_dir / "county_year_summary.json") as f:
    county_summary = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_month.json") as f:
    county_month = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_hour.json") as f:
    county_hour = pd.DataFrame(json.load(f))

print(f"County summary records: {len(county_summary):,}")

In [ ]:
# View utility-related columns
print("Utility-related columns:")
print(county_summary[['utility_provider', 'market_type']].head(10))

In [ ]:
# Unique utilities and market types
print(f"Unique utility providers: {county_summary['utility_provider'].nunique()}")
print(f"Unique market types: {county_summary['market_type'].nunique()}")
print(f"\nMarket types: {county_summary['market_type'].unique()}")

## Utility-Level Yearly Statistics

Aggregate outage data by utility provider.

In [ ]:
# Aggregate by utility provider
utility_totals = county_summary.groupby('utility_provider').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum',
    'fips': 'nunique',
    'market_type': 'first'  # Get associated market type
}).rename(columns={'fips': 'counties_count'}).reset_index()

# Calculate customer hours per capita
utility_totals['customer_hours_per_capita'] = utility_totals['customer_hours'] / utility_totals['population']

print(f"Total utilities: {len(utility_totals)}")
utility_totals.head(10)

In [ ]:
# Descriptive statistics for utility-level data
utility_totals[['outage_count', 'customer_hours', 'customer_hours_per_capita', 'counties_count']].describe()

In [ ]:
# Top 20 utilities by outage count
top_utilities_outages = utility_totals.nlargest(20, 'outage_count')

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(top_utilities_outages['utility_provider'][::-1], 
               top_utilities_outages['outage_count'][::-1], 
               color='steelblue')
ax.set_title('Top 20 Utilities by Total Outage Count')
ax.set_xlabel('Outage Count')
ax.set_ylabel('Utility Provider')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 utilities by customer hours
top_utilities_hours = utility_totals.nlargest(20, 'customer_hours')

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top_utilities_hours['utility_provider'][::-1], 
        top_utilities_hours['customer_hours'][::-1] / 1e6, 
        color='coral')
ax.set_title('Top 20 Utilities by Total Customer Hours')
ax.set_xlabel('Customer Hours (millions)')
ax.set_ylabel('Utility Provider')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 utilities by customer hours per capita (min population threshold)
min_pop = 100000  # At least 100k population served
utility_filtered = utility_totals[utility_totals['population'] >= min_pop]
top_utilities_per_capita = utility_filtered.nlargest(20, 'customer_hours_per_capita')

fig, ax = plt.subplots(figsize=(12, 10))
ax.barh(top_utilities_per_capita['utility_provider'][::-1], 
        top_utilities_per_capita['customer_hours_per_capita'][::-1], 
        color='teal')
ax.set_title(f'Top 20 Utilities by Customer Hours per Capita\n(min population: {min_pop:,})')
ax.set_xlabel('Customer Hours per Capita')
ax.set_ylabel('Utility Provider')
plt.tight_layout()
plt.show()

### Distribution of Utility-Level Metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Outage count distribution
axes[0, 0].hist(utility_totals['outage_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Utility Outage Count')
axes[0, 0].set_xlabel('Outage Count')
axes[0, 0].set_ylabel('Frequency')

axes[1, 0].hist(np.log10(utility_totals['outage_count'] + 1), bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Utility Outage Count (log10)')
axes[1, 0].set_xlabel('log10(Outage Count)')
axes[1, 0].set_ylabel('Frequency')

# Customer hours distribution
axes[0, 1].hist(utility_totals['customer_hours'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Utility Customer Hours')
axes[0, 1].set_xlabel('Customer Hours')
axes[0, 1].set_ylabel('Frequency')

axes[1, 1].hist(np.log10(utility_totals['customer_hours'] + 1), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_title('Distribution of Utility Customer Hours (log10)')
axes[1, 1].set_xlabel('log10(Customer Hours)')
axes[1, 1].set_ylabel('Frequency')

# Counties served distribution
axes[0, 2].hist(utility_totals['counties_count'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0, 2].set_title('Distribution of Counties Served')
axes[0, 2].set_xlabel('Number of Counties')
axes[0, 2].set_ylabel('Frequency')

axes[1, 2].hist(np.log10(utility_totals['counties_count'] + 1), bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[1, 2].set_title('Distribution of Counties Served (log10)')
axes[1, 2].set_xlabel('log10(Number of Counties)')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Percentile analysis for utility metrics
percentiles = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
metrics = ['outage_count', 'customer_hours', 'customer_hours_per_capita', 'counties_count']

percentile_df = utility_totals[metrics].quantile(percentiles)
percentile_df.index = [f"{int(p*100)}%" for p in percentiles]
print("Percentile distribution for utility metrics:")
percentile_df

## Market Type Analysis

Compare outage statistics across different electricity market structures.

In [ ]:
# Aggregate by market type
market_totals = county_summary.groupby('market_type').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum',
    'fips': 'nunique',
    'utility_provider': 'nunique'
}).rename(columns={'fips': 'counties_count', 'utility_provider': 'utilities_count'})

market_totals['customer_hours_per_capita'] = market_totals['customer_hours'] / market_totals['population']

print("Outage statistics by market type:")
market_totals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Outage count by market type
axes[0, 0].bar(market_totals.index, market_totals['outage_count'], color='steelblue')
axes[0, 0].set_title('Total Outages by Market Type')
axes[0, 0].set_xlabel('Market Type')
axes[0, 0].set_ylabel('Outage Count')
axes[0, 0].tick_params(axis='x', rotation=45)

# Customer hours by market type
axes[0, 1].bar(market_totals.index, market_totals['customer_hours'] / 1e6, color='coral')
axes[0, 1].set_title('Total Customer Hours by Market Type')
axes[0, 1].set_xlabel('Market Type')
axes[0, 1].set_ylabel('Customer Hours (millions)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Customer hours per capita by market type
axes[1, 0].bar(market_totals.index, market_totals['customer_hours_per_capita'], color='teal')
axes[1, 0].set_title('Customer Hours per Capita by Market Type')
axes[1, 0].set_xlabel('Market Type')
axes[1, 0].set_ylabel('Customer Hours per Capita')
axes[1, 0].tick_params(axis='x', rotation=45)

# Number of utilities by market type
axes[1, 1].bar(market_totals.index, market_totals['utilities_count'], color='purple')
axes[1, 1].set_title('Number of Utilities by Market Type')
axes[1, 1].set_xlabel('Market Type')
axes[1, 1].set_ylabel('Number of Utilities')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Box plots comparing distributions across market types
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Outage count distribution by market type
county_summary.boxplot(column='outage_count', by='market_type', ax=axes[0])
axes[0].set_title('Outage Count Distribution by Market Type')
axes[0].set_xlabel('Market Type')
axes[0].set_ylabel('Outage Count')
plt.suptitle('')

# Log scale version
county_summary_log = county_summary.copy()
county_summary_log['log_customer_hours'] = np.log10(county_summary_log['customer_hours'] + 1)
county_summary_log.boxplot(column='log_customer_hours', by='market_type', ax=axes[1])
axes[1].set_title('Customer Hours (log10) by Market Type')
axes[1].set_xlabel('Market Type')
axes[1].set_ylabel('log10(Customer Hours)')
plt.suptitle('')

county_summary_log['log_per_capita'] = np.log10(county_summary_log['customer_hours_per_capita'] + 0.001)
county_summary_log.boxplot(column='log_per_capita', by='market_type', ax=axes[2])
axes[2].set_title('Customer Hours per Capita (log10) by Market Type')
axes[2].set_xlabel('Market Type')
axes[2].set_ylabel('log10(Customer Hours per Capita)')
plt.suptitle('')

plt.tight_layout()
plt.show()

## Utility Yearly Trends

In [ ]:
# Aggregate by utility and year
utility_yearly = county_summary.groupby(['utility_provider', 'year']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'population': 'sum',
    'market_type': 'first'
}).reset_index()

utility_yearly['customer_hours_per_capita'] = utility_yearly['customer_hours'] / utility_yearly['population']

print(f"Utility-year combinations: {len(utility_yearly)}")
utility_yearly.head(10)

In [ ]:
# Get top 5 utilities by total outages and show yearly trends
top_5_utilities = utility_totals.nlargest(5, 'outage_count')['utility_provider'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for utility in top_5_utilities:
    utility_data = utility_yearly[utility_yearly['utility_provider'] == utility]
    axes[0].plot(utility_data['year'], utility_data['outage_count'], marker='o', label=utility)
    axes[1].plot(utility_data['year'], utility_data['customer_hours'] / 1e6, marker='o', label=utility)

axes[0].set_title('Yearly Outage Trend - Top 5 Utilities')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Outage Count')
axes[0].legend(fontsize=8)

axes[1].set_title('Yearly Customer Hours Trend - Top 5 Utilities')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Customer Hours (millions)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Seasonality by Market Type

In [ ]:
# Merge county_month with county_summary to get market type
county_month_enriched = county_month.merge(
    county_summary[['fips', 'year', 'market_type']].drop_duplicates(),
    on=['fips', 'year'],
    how='left'
)

# Aggregate by market type and month
market_monthly = county_month_enriched.groupby(['market_type', 'month']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
market_monthly['month_name'] = market_monthly['month'].apply(lambda x: month_names[x-1])

market_monthly.head()

In [ ]:
# Pivot for plotting
market_monthly_pivot = market_monthly.pivot(index='month', columns='market_type', values='outage_count')

fig, ax = plt.subplots(figsize=(12, 6))
market_monthly_pivot.plot(kind='bar', ax=ax)
ax.set_title('Monthly Outages by Market Type')
ax.set_xlabel('Month')
ax.set_ylabel('Outage Count')
ax.set_xticklabels(month_names, rotation=45)
ax.legend(title='Market Type')
plt.tight_layout()
plt.show()

In [ ]:
# Normalize by market type total to see seasonal patterns
market_totals_monthly = market_monthly.groupby('market_type')['outage_count'].transform('sum')
market_monthly['outage_pct'] = market_monthly['outage_count'] / market_totals_monthly * 100

market_monthly_pct_pivot = market_monthly.pivot(index='month', columns='market_type', values='outage_pct')

fig, ax = plt.subplots(figsize=(12, 6))
for col in market_monthly_pct_pivot.columns:
    ax.plot(range(1, 13), market_monthly_pct_pivot[col], marker='o', label=col)

ax.set_title('Seasonal Pattern by Market Type (% of Annual Total)')
ax.set_xlabel('Month')
ax.set_ylabel('Percentage of Annual Outages')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.legend(title='Market Type')
plt.tight_layout()
plt.show()

## Time of Day by Market Type

In [ ]:
# Merge county_hour with county_summary to get market type
county_hour_enriched = county_hour.merge(
    county_summary[['fips', 'year', 'market_type']].drop_duplicates(),
    on=['fips', 'year'],
    how='left'
)

# Aggregate by market type and hour
market_hourly = county_hour_enriched.groupby(['market_type', 'hour']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

market_hourly.head()

In [ ]:
# Normalize by market type total
market_totals_hourly = market_hourly.groupby('market_type')['outage_count'].transform('sum')
market_hourly['outage_pct'] = market_hourly['outage_count'] / market_totals_hourly * 100

market_hourly_pct_pivot = market_hourly.pivot(index='hour', columns='market_type', values='outage_pct')

fig, ax = plt.subplots(figsize=(14, 6))
for col in market_hourly_pct_pivot.columns:
    ax.plot(market_hourly_pct_pivot.index, market_hourly_pct_pivot[col], marker='o', label=col)

ax.set_title('Hourly Pattern by Market Type (% of Total)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Percentage of Outages')
ax.set_xticks(range(0, 24))
ax.legend(title='Market Type')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of outages by hour and market type
market_hourly_pivot = market_hourly.pivot(index='market_type', columns='hour', values='outage_count')

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(market_hourly_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, annot_kws={'size': 8})
ax.set_title('Outage Count by Market Type and Hour')
ax.set_xlabel('Hour')
ax.set_ylabel('Market Type')
plt.tight_layout()
plt.show()